In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib as plt
from shared import init_analyse  #initiële analyse functie
#Importeren van libraries

In [6]:
#%%toc

## Analyse van sales data

In dit onderdeel importeren en analyseren we de salesdata met behulp van pandas

### 1. Importeren en inlezen dataset

In [7]:
sales_path= Path(r"Data\Contoso_Sales.csv")
#Inlezen sales.csv
sales_df = pd.read_csv(sales_path,
                       decimal=",")
#Dataframe koppelen aan variabele
#Decimal argument meegegeven, omdat pandas standaard . als delimiter ziet.

Data is met behulp van pathlib en pandas ingelezen.  
Bij het inlezen komt er een dtypewarning voor de kolommen ReturnAmount en DiscountAmount.  
Dit betekent dat hier verschillende soorten type data in voorkomen, dus niet alleen bedragen.

### 2. Eerste analyse
Hier wordt de algehele dataset geanalyseerd op afwijkingen, waar hierna op groter detailniveau naar gekeken wordt.

In [8]:
sales_df.info()
#Eerste data over kolommen opvragen

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   SalesKey          int64  
 1   DateKey           str    
 2   channelKey        int64  
 3   StoreKey          int64  
 4   ProductKey        int64  
 5   PromotionKey      int64  
 6   UnitCost          float64
 7   UnitPrice         float64
 8   SalesQuantity     int64  
 9   ReturnQuantity    int64  
 10  ReturnAmount      float64
 11  DiscountQuantity  int64  
 12  DiscountAmount    float64
 13  TotalCost         float64
 14  SalesAmount       float64
dtypes: float64(6), int64(8), str(1)
memory usage: 261.2 MB


- Datekey wordt als string weergegeven, deze dient nog omgezet te worden naar een tijdreeks.
- UnitCost, UnitPrice, TotalCost en SalesAmount worden als strings geïnterpreteerd, verwachting is dat dit floats zouden moeten zijn.
- ReturnAmount en DiscountAmount zijn objects, onderzoeken wat voor waarden dit zijn. Hangt ook samen met de eerdere dtypewarning bij het inlezen.

In [9]:
sales_df.head(20)
#De eerste 20 rijn tonen met waarden.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,"2011-05-07 00:00:00,000",1,76,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
1,26527,"2013-05-22 00:00:00,000",1,26,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
2,26702,"2013-09-30 00:00:00,000",1,298,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
3,33125,"2012-10-31 00:00:00,000",1,264,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
4,52306,"2013-06-26 00:00:00,000",1,98,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
5,67230,"2012-10-28 00:00:00,000",1,194,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
6,79246,"2012-05-12 00:00:00,000",1,230,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
7,87210,"2013-04-17 00:00:00,000",1,84,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
8,91129,"2013-05-26 00:00:00,000",1,215,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
9,109148,"2013-04-06 00:00:00,000",1,93,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0


Datekey lijkt een datumwaarde te moeten zijn, deze moet bij het opschonen van de data omgezet worden.  
UnitCost, UnitPrice, TotalCost en SalesAmount lijkt om bedragen te gaan. De stuksprijs maal de quantity is hier ook gelijk aan de Totalcost en SalesAmount.  
Gezien er geen strings in de tabel zitten ook geen noodzaak om te controleren of deze afwijken van elkaar in schrijfwijze.

In [10]:
sales_df.tail(20)
#Laatste 20 waarden bekijken.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
2282462,3366917,"2011-06-28 00:00:00,000",1,120,613,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282463,3367028,"2011-05-20 00:00:00,000",1,12,320,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282464,3369192,"2012-05-15 00:00:00,000",1,60,320,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282465,3369889,"2013-04-29 00:00:00,000",1,167,201,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282466,3369908,"2012-10-06 00:00:00,000",1,115,405,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282467,3370934,"2012-04-21 00:00:00,000",1,142,624,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282468,3371111,"2013-05-23 00:00:00,000",1,21,345,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282469,3372220,"2011-05-29 00:00:00,000",1,2,613,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282470,3378122,"2011-06-25 00:00:00,000",1,243,405,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282471,3378463,"2013-10-26 00:00:00,000",1,282,345,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0


Laatste 20 rijen laten hetzelfde patroon zien. In deze rijen zitten ook returns verwerkt. Deze lijken wel goed verwerkt te worden in de TotalCost, bijvoorbeeld 10 items verkocht - 1 return = 9* unitcost.  
Op salesniveau lijkt dit echter niet correct doorberekend. De SalesAmount blijft gelijk aan 10 maal de UnitPrice, ondanks de return.  
Onderzoeken of dit elders wordt opgevangen. 

In [11]:
sales_df.describe()

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
count,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06,2.282482e+06
mean,1.703424e+06,1.627063e+00,1.963849e+02,8.811926e+02,8.498340e+00,1.245984e+02,2.971792e+02,1.616679e+01,1.429619e-01,4.256032e+01,1.476230e+00,5.551586e+01,1.556379e+03,3.654454e+03
std,9.831530e+05,1.028135e+00,9.553847e+01,5.094803e+02,8.258284e+00,1.267489e+02,3.473084e+02,3.935030e+01,3.599263e-01,1.720155e+02,1.798820e+00,1.390498e+02,1.888546e+03,4.853793e+03
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,4.800000e-01,9.500000e-01,4.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.920000e+00,3.040000e+00
25%,8.518095e+05,1.000000e+00,1.210000e+02,4.530000e+02,1.000000e+00,3.058000e+01,6.600000e+01,9.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.214800e+02,6.584220e+02
50%,1.704424e+06,1.000000e+00,2.000000e+02,8.800000e+02,4.000000e+00,8.737000e+01,2.000000e+02,1.000000e+01,0.000000e+00,0.000000e+00,1.000000e+00,2.415000e+00,9.860400e+02,2.293600e+03
75%,2.554565e+06,2.000000e+00,2.960000e+02,1.273000e+03,1.400000e+01,1.626400e+02,3.580000e+02,1.300000e+01,0.000000e+00,0.000000e+00,3.000000e+00,5.398000e+01,2.065230e+03,4.699700e+03
max,3.406089e+06,4.000000e+00,3.100000e+02,2.517000e+03,2.800000e+01,9.608200e+02,2.899990e+03,2.880000e+03,5.000000e+00,9.996000e+03,2.400000e+01,6.497400e+03,8.411520e+04,2.354670e+05


In [12]:
sales_df.isna().sum()
#Controleren op lege waarden in cellen. Dit is niet van toepassing.
sales_df.nunique() 
#Aantal unique waarden opsommen.

SalesKey            2282482
DateKey                1096
channelKey                4
StoreKey                306
ProductKey             1689
PromotionKey             28
UnitCost                397
UnitPrice               350
SalesQuantity            98
ReturnQuantity            6
ReturnAmount            782
DiscountQuantity         23
DiscountAmount         8572
TotalCost             14114
SalesAmount           51717
dtype: int64

Controles om te kruisen met andere tabellen:
- 4 verkoopkanalen volgens sales set.
- 306 unieke winkels volgens sales set.
- 1689 soort producten
- 28 promoties
- 783 retourregels

### 3. Corrigeren datatypes

Op basis van de analyse zien we dat de datum gegevens nu niet als juiste dtype worden ingegeven.  
In dit onderdeel zetten we deze om naar bruikbare waarden in de dataset.

In [13]:
sales_df2 = sales_df.copy()
#Dataframe kopieeëren als backup 
sales_df2

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,"2011-05-07 00:00:00,000",1,76,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
1,26527,"2013-05-22 00:00:00,000",1,26,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
2,26702,"2013-09-30 00:00:00,000",1,298,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
3,33125,"2012-10-31 00:00:00,000",1,264,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
4,52306,"2013-06-26 00:00:00,000",1,98,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,"2013-05-15 00:00:00,000",1,40,397,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282478,3395082,"2013-10-04 00:00:00,000",1,52,565,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282479,3396997,"2011-07-26 00:00:00,000",1,303,381,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282480,3398163,"2013-07-04 00:00:00,000",1,247,201,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0


In [14]:
sales_df2["DateKey"] = pd.to_datetime(sales_df2["DateKey"], format="%Y-%m-%d %H:%M:%S,%f")
#Omzetten naar datetime van DateKey
sales_df2

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,2011-05-07,1,76,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
1,26527,2013-05-22,1,26,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
2,26702,2013-09-30,1,298,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
3,33125,2012-10-31,1,264,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
4,52306,2013-06-26,1,98,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.40,6990.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,2013-05-15,1,40,397,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282478,3395082,2013-10-04,1,52,565,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282479,3396997,2011-07-26,1,303,381,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0
2282480,3398163,2013-07-04,1,247,201,1,321.44,699.0,10,1,699.0,0,0.0,2892.96,6990.0


In [15]:
print(sales_df2.loc[sales_df2["DateKey"].idxmin(),"DateKey"])
print(sales_df2.loc[sales_df2["DateKey"].idxmax(),"DateKey"])
#Kijken naar begin en einddatum van dataset.

2011-01-01 00:00:00
2013-12-31 00:00:00


Dataset loopt van 1 januari 2011 tot en met 31 december 2013

In [16]:
datesales = sales_df2.set_index("DateKey")

In [17]:
datesales.resample("ME").agg(aantal_keys=("SalesKey","nunique"),
                             verkoopkanalen=("channelKey","nunique"),
                             kosten=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum")
                             )

,aantal_keys,verkoopkanalen,kosten,SalesAmount
DateKey,,,,
2011-01-31,73868,4,8.159359e+07,1.933056e+08
2011-02-28,70212,4,8.841217e+07,2.094391e+08
2011-03-31,73596,4,8.690551e+07,2.039920e+08
2011-04-30,91338,4,1.168721e+08,2.768910e+08
2011-05-31,94223,4,1.222369e+08,2.887495e+08
2011-06-30,92553,4,1.198062e+08,2.831866e+08
2011-07-31,76193,4,1.153630e+08,2.728186e+08
2011-08-31,76085,4,1.115093e+08,2.637803e+08
2011-09-30,74802,4,1.088589e+08,2.572828e+08


In [18]:
sales_df2.isna().sum()

SalesKey            0
DateKey             0
channelKey          0
StoreKey            0
ProductKey          0
PromotionKey        0
UnitCost            0
UnitPrice           0
SalesQuantity       0
ReturnQuantity      0
ReturnAmount        0
DiscountQuantity    0
DiscountAmount      0
TotalCost           0
SalesAmount         0
dtype: int64

In [19]:
init_analyse(sales_df2)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 15 columns):
 #   Column            Dtype         
---  ------            -----         
 0   SalesKey          int64         
 1   DateKey           datetime64[us]
 2   channelKey        int64         
 3   StoreKey          int64         
 4   ProductKey        int64         
 5   PromotionKey      int64         
 6   UnitCost          float64       
 7   UnitPrice         float64       
 8   SalesQuantity     int64         
 9   ReturnQuantity    int64         
 10  ReturnAmount      float64       
 11  DiscountQuantity  int64         
 12  DiscountAmount    float64       
 13  TotalCost         float64       
 14  SalesAmount       float64       
dtypes: datetime64[us](1), float64(6), int64(8)
memory usage: 261.2 MB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


SalesKey            0
DateKey             0
channelKey          0
StoreKey            0
ProductKey          0
PromotionKey        0
UnitCost            0
UnitPrice           0
SalesQuantity       0
ReturnQuantity      0
ReturnAmount        0
DiscountQuantity    0
DiscountAmount      0
TotalCost           0
SalesAmount         0
dtype: int64


Eerste 20 rijen:


,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,2011-05-07,1,76,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
1,26527,2013-05-22,1,26,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
2,26702,2013-09-30,1,298,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
3,33125,2012-10-31,1,264,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
4,52306,2013-06-26,1,98,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
5,67230,2012-10-28,1,194,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
6,79246,2012-05-12,1,230,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
7,87210,2013-04-17,1,84,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
8,91129,2013-05-26,1,215,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
9,109148,2013-04-06,1,93,375,1,321.44,699.0,10,0,0.0,0,0.0,3214.4,6990.0
